# tensor-item-scalar — ex8: random walk histogram via .item()

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-item-scalar`. Running the final beacon cell reports progress against the `Numpy: Core array literacy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Core array literacy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-item-scalar`** (exercise 8). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-item-scalar"
DD_SUBTOPIC = "Numpy: Core array literacy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch `.item()` — quick refresher

`tensor.item()` extracts a Python scalar (`int`, `float`, or `bool`) from a 0-D or 1-element tensor. It detaches from the autograd graph and pulls the value back to CPU. This is the canonical bridge from tensor-world to Python-world: logging, control flow, plotting, and stop conditions all need scalars.

**Calling `.item()` on a multi-element tensor raises.** Use `.tolist()` if you want every element as a Python list. Calling `.item()` inside a hot inner loop forces a CPU sync — fine for diagnostics, expensive in the training step itself.

### Exercise 8 — random walk histogram via .item()

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Run a stochastic simulation that accumulates scalar tensors via `.item()` into a Python list, then visualize the empirical distribution as a histogram.
> Keywords: random-walk, simulation, histogram, visualization
> ```

**KCs targeted:** `item-zero-d-extract`, `item-vs-tolist`

Implement `ex8_random_walks(n_walks, n_steps, generator)`. A Monte-Carlo simulation that combines tensor ops with Python-scalar accumulation:

1. Run `n_walks` independent 1-D random walks of `n_steps` each. Per walk: start at `0.0`, repeatedly add a sample from the standard normal distribution.
2. Implementation: compute all increments at once with `t.randn(n_walks, n_steps, generator=generator)`, sum along the step axis to get a `(n_walks,)` tensor of final positions.
3. **For each walk, call `.item()` on its final position** and append to a Python list `final_positions`. (Bulk `.tolist()` would also work — see solution notes — but the per-walk `.item()` loop is the version this exercise drills.)
4. Return a dict: `{'final_positions': list_of_floats, 'sample_walks': (5, n_steps) tensor of cumulative paths for the first 5 walks}`.

Inputs:
- `n_walks`: int.
- `n_steps`: int.
- `generator`: `torch.Generator` (for reproducibility — pass it through to `t.randn`).

The visualization is a two-panel plot: trajectories of the 5 sample walks (left), histogram of `final_positions` overlaid with the theoretical N(0, √n_steps) curve (right).

In [ ]:
def ex8_random_walks(n_walks: int, n_steps: int, generator: t.Generator) -> dict:
    """Simulate random walks; extract per-walk final position via .item()."""
    raise NotImplementedError()


def _test_ex8():
    rng = t.Generator().manual_seed(2025)
    result = ex8_random_walks(n_walks=500, n_steps=100, generator=rng)
    assert isinstance(result, dict)
    assert set(result.keys()) >= {'final_positions', 'sample_walks'}, f'missing keys: {result.keys()}'
    fp = result['final_positions']
    assert isinstance(fp, list), f'final_positions must be a Python list, got {type(fp)}'
    assert len(fp) == 500, f'expected 500 entries, got {len(fp)}'
    for i, v in enumerate(fp[:10]):
        assert isinstance(v, float), f'final_positions[{i}] must be a Python float (from .item()), got {type(v)}'
    # Statistical sanity — mean ≈ 0, std ≈ sqrt(n_steps) = 10.
    import statistics as _stats
    mean = _stats.mean(fp)
    stdev = _stats.stdev(fp)
    assert abs(mean) < 1.5, f'expected mean near 0, got {mean:.3f}'
    assert 8.0 < stdev < 12.0, f'expected stdev near 10, got {stdev:.3f}'
    # sample_walks shape check.
    sw = result['sample_walks']
    assert isinstance(sw, Tensor)
    assert sw.shape == (5, 100), f'expected (5, 100), got {tuple(sw.shape)}'
    # Cumulative paths must start near 0 (first step) and drift from there.
    # Each row's last value must match its final_position entry (within fp accuracy).
    for i in range(5):
        assert abs(sw[i, -1].item() - fp[i]) < 1e-4, (
            f'sample_walks[{i}, -1] = {sw[i, -1].item()} but final_positions[{i}] = {fp[i]}'
        )

    # --- Two-panel visualization ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    for i in range(5):
        ax1.plot(sw[i].numpy(), alpha=0.7)
    ax1.set_title('ex8 — first 5 sample walks')
    ax1.set_xlabel('step')
    ax1.set_ylabel('position')
    ax1.grid(True, alpha=0.3)

    ax2.hist(fp, bins=30, density=True, color='lightcoral', edgecolor='black', label='empirical')
    # Theoretical N(0, sqrt(n_steps)) density.
    xs_th = np.linspace(min(fp), max(fp), 200)
    sigma = 10.0
    ys_th = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * (xs_th / sigma) ** 2)
    ax2.plot(xs_th, ys_th, 'navy', linewidth=2, label='N(0, √100)')
    ax2.set_title('ex8 — final positions vs theory')
    ax2.set_xlabel('final position (extracted via .item())')
    ax2.set_ylabel('density')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex8')
    print("ex8 ✓")

_test_ex8()

<details><summary>Solution</summary>

```python
def ex8_random_walks(n_walks: int, n_steps: int, generator: t.Generator) -> dict:
    increments = t.randn(n_walks, n_steps, generator=generator)
    cumulative = increments.cumsum(dim=1)  # (n_walks, n_steps)
    final_tensor = cumulative[:, -1]       # (n_walks,)
    final_positions = []
    for i in range(n_walks):
        final_positions.append(final_tensor[i].item())
    return {
        'final_positions': final_positions,
        'sample_walks': cumulative[:5].clone(),
    }
```

**`.item()` in a loop vs `.tolist()`.** For pulling many scalars back to Python, `final_tensor.tolist()` is faster (one CPU sync instead of `n_walks` of them). The per-walk `.item()` loop here is deliberately the slow version — it's the right shape for cases where each walk's scalar feeds an immediate Python-side decision (e.g. branching, accumulating into a non-tensor structure).

**Cumulative-sum trick.** A random walk is just the cumulative sum of iid normal increments. Generating all the noise in one `t.randn(n_walks, n_steps)` call and doing one `.cumsum(dim=1)` is orders of magnitude faster than a Python `for step in range` loop, even though it produces the same trajectories.

**Why the empirical std ≈ √n_steps.** A standard random walk's variance grows linearly with the number of steps, so its standard deviation grows as √n. With `n_steps=100` the theoretical std is exactly 10.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex8',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()